# FUT.AI — Homografía manual (MVP local)

**Objetivo:** partir del MP4 y del CSV de seguimiento existente para calibrar **un fotograma** y proyectar los jugadores a un plano 2D en metros. Sin nuevas inferencias de YOLO.

**Procedimiento:** 1) configurar las rutas; 2) elegir fotograma con **≥4 referencias conocidas y no colineales**; 3) hacer clic sucesivamente en el fotograma y en el plano, sobre **el mismo punto físico**; 4) validar la homografía; 5) exportar coordenadas y matriz.

> **Límite importante:** este notebook usa una **homografía fija de un fotograma**. Si la cámara gira, se desplaza o hace zoom, esta matriz deja de ser válida. **No calcules distancias ni velocidades del video completo con ella.** Más adelante implementaremos la actualización temporal de la homografía (keypoints del terreno u otra técnica validada).

**Cancha de referencia:** 105 m de largo × 68 m de ancho. Cambia las dimensiones si conoces las reales. Los landmarks elegidos deben ser **puntos geométricos de la cancha**, no posiciones arbitrarias de jugadores.

Puntos útiles: intersección de la línea media con las bandas, punto central, intersecciones del círculo central con la línea media, esquinas del área grande y del área pequeña, siempre que realmente se distingan. Elige puntos repartidos en **dos dimensiones**, no cuatro puntos sobre la línea media.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# CAMBIA ESTAS RUTAS para que coincidan con tu sesión de Colab
VIDEO = Path('/content/prueba2_20s.mp4')
CSV_TRACKS = Path('/content/futai/diagnostico/prueba2_20s_yolo11s.csv')
SEGUNDO_CALIBRACION = 5.0  # Cambia el segundo hasta ver >=4 referencias conocidas
LARGO_M = 105.0
ANCHO_M = 68.0
SALIDA = Path('/content/futai/homografia')
SALIDA.mkdir(parents=True, exist_ok=True)

if not VIDEO.is_file():
    raise FileNotFoundError(f'Falta el video: {VIDEO}. Súbelo a Colab o corrige VIDEO.')
if not CSV_TRACKS.is_file():
    raise FileNotFoundError(f'Falta el CSV: {CSV_TRACKS}. Corrige CSV_TRACKS.')

tracks = pd.read_csv(CSV_TRACKS)
columnas = {'frame','tiempo_s','track_id','confianza','x1','y1','x2','y2'}
if not columnas.issubset(tracks.columns):
    raise ValueError(f'Columnas faltantes: {columnas-set(tracks.columns)}')

cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise ValueError('El video no se puede abrir')
fps = float(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if fps <= 0:
    cap.release()
    raise ValueError('FPS inválido')
frame_idx = max(0, min(round(SEGUNDO_CALIBRACION*fps), total_frames-1))
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise ValueError('No se pudo leer el fotograma elegido')

alto_original, ancho_original = frame_bgr.shape[:2]
if tracks[['x2','y2']].max().iloc[0] > ancho_original+5 or tracks[['x2','y2']].max().iloc[1] > alto_original+5:
    raise ValueError('CSV y video no tienen el mismo sistema de coordenadas. Verifica que correspondan al mismo archivo.')
print(f'Video: {VIDEO.name} | {ancho_original}x{alto_original} | {fps:.2f} FPS')
print(f'Calibración: frame {frame_idx}, t = {frame_idx/fps:.2f} segundos')
print(f'CSV: {len(tracks):,} observaciones, {tracks.track_id.nunique()} tracks')

In [ ]:
# Plano esquemático de referencia, en escala métrica (no una medición del estadio).
def construir_plano(largo, ancho, escala=6, margen=42):
    w, h = int(round(largo*escala+2*margen)), int(round(ancho*escala+2*margen))
    im = np.full((h,w,3), (44,116,63), dtype=np.uint8)

    def p(x,y):
        return (int(round(margen+x*escala)), int(round(margen+y*escala)))
    blanco=(245,245,245)
    cv2.rectangle(im, p(0,0), p(largo,ancho), blanco, 2)
    cv2.line(im,p(largo/2,0),p(largo/2,ancho),blanco,2)
    cv2.circle(im,p(largo/2,ancho/2),int(round(9.15*escala)),blanco,2)
    cv2.circle(im,p(largo/2,ancho/2),4,blanco,-1)
    for izq in (True,False):
        x_area = 16.5 if izq else largo-16.5
        x_meta = 0 if izq else largo
        y1,y2=(ancho-40.32)/2,(ancho+40.32)/2
        cv2.rectangle(im,p(min(x_meta,x_area),y1),p(max(x_meta,x_area),y2),blanco,2)
        x_chica = 5.5 if izq else largo-5.5
        ym1,ym2=(ancho-18.32)/2,(ancho+18.32)/2
        cv2.rectangle(im,p(min(x_meta,x_chica),ym1),p(max(x_meta,x_chica),ym2),blanco,2)
        x_pen = 11 if izq else largo-11
        cv2.circle(im,p(x_pen,ancho/2),3,blanco,-1)
    # Ejes: x a lo largo del campo desde la portería izquierda, y de banda superior a inferior.
    cv2.putText(im, '0 m',p(0,0),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)
    cv2.putText(im, f'{largo:g} m',p(largo-10,0),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)
    return im, escala, margen

plano_bgr, ESCALA_PLANO, MARGEN_PLANO = construir_plano(LARGO_M,ANCHO_M)
fig,ax=plt.subplots(2,1,figsize=(15,11))
ax[0].imshow(cv2.cvtColor(frame_bgr,cv2.COLOR_BGR2RGB));ax[0].set_title('Fotograma de calibración: busca referencias pintadas en el césped');ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(plano_bgr,cv2.COLOR_BGR2RGB));ax[1].set_title('Plano métrico: x = 0 a 105 m, y = 0 a 68 m');ax[1].axis('off')
plt.tight_layout();plt.show()

## 3. Selección interactiva de correspondencias en Colab

1. Clic en una **referencia geométrica** del fotograma (p. ej., la unión entre línea media y banda).
2. Clic en **ese mismo punto** del plano verde de abajo.
3. Repite con 4–8 puntos que **no queden todos alineados**. Cuanto más dispersos, mejor.
4. Pulsa **Terminar**. Puedes deshacer el último par de puntos si te equivocas.

**Regla clave:** no debes seleccionar un jugador en la imagen y su posición aproximada en el plano. Solo líneas, intersecciones o puntos con coordenadas conocidas.

La interfaz utiliza JavaScript de Google Colab para permitir clics: **no funcionará en el Jupyter Notebook local tradicional**. En ese caso, rellena manualmente `PARES` en la celda alternativa.

In [ ]:
from IPython.display import display
from google.colab.output import eval_js
import base64

# Redimensionamos SOLO para la interfaz; los clics se convierten a los píxeles originales.
ESCALA_VISUAL = min(1.0, 1100/ancho_original)
frame_muestra = cv2.resize(frame_bgr, (round(ancho_original*ESCALA_VISUAL), round(alto_original*ESCALA_VISUAL)))
_, buf_frame = cv2.imencode('.jpg',frame_muestra,[cv2.IMWRITE_JPEG_QUALITY,90])
_, buf_plano = cv2.imencode('.png',plano_bgr)
url_frame = 'data:image/jpeg;base64,'+base64.b64encode(buf_frame).decode('ascii')
url_plano = 'data:image/png;base64,'+base64.b64encode(buf_plano).decode('ascii')

js = r"""new Promise((resolve,reject)=>{
  const VIDEO_URL = "__URL_FRAME__";
  const PITCH_URL = "__URL_PLANO__";
  const ORIGINAL_W = __ORIGINAL_W__;
  const ORIGINAL_H = __ORIGINAL_H__;
  const PAD = __PAD__;
  const SCALE = __SCALE__;
  const LARGO = __LARGO__;
  const ANCHO = __ANCHO__;
  const root = document.createElement('div');
  root.style='background:#182323;color:white;padding:14px;border-radius:10px;font:14px sans-serif;max-width:1150px';
  root.innerHTML='<h3>FUT.AI — Selección de puntos</h3><p id="estado_click">Paso 1: clic en una referencia del FOTOGRAMA.</p>';
  const cv = document.createElement('canvas');
  const cp = document.createElement('canvas');
  for(const c of [cv,cp]){c.style='display:block;max-width:100%;cursor:crosshair;border:1px solid #777;margin:8px 0';root.appendChild(c);}
  const bar = document.createElement('div');bar.style='display:flex;gap:10px;align-items:center';root.appendChild(bar);
  const undo=document.createElement('button');undo.textContent='Deshacer último punto';bar.appendChild(undo);
  const reset=document.createElement('button');reset.textContent='Reiniciar';bar.appendChild(reset);
  const finish=document.createElement('button');finish.textContent='Terminar (mín. 4 pares)';finish.disabled=true;bar.appendChild(finish);
  document.body.appendChild(root);
  const estado=root.querySelector('#estado_click');
  let pares=[], pendiente=null;
  const vi=new Image(),pi=new Image();
  vi.onload=()=>{cv.width=vi.naturalWidth;cv.height=vi.naturalHeight;render();};
  pi.onload=()=>{cp.width=pi.naturalWidth;cp.height=pi.naturalHeight;render();};
  vi.src=VIDEO_URL;pi.src=PITCH_URL;
  function pos(e,canvas){
    const rect=canvas.getBoundingClientRect();
    return [(e.clientX-rect.left)*canvas.width/rect.width,(e.clientY-rect.top)*canvas.height/rect.height];
  }
  function marker(ctx,x,y,n){
    ctx.beginPath();ctx.arc(x,y,5,0,2*Math.PI);ctx.fillStyle='#ff3366';ctx.fill();
    ctx.font='bold 16px sans-serif';ctx.fillStyle='#fff';ctx.fillText(String(n),x+7,y-6);
  }
  function render(){
    if(vi.complete && vi.naturalWidth && cv.width){const ctx=cv.getContext('2d');ctx.drawImage(vi,0,0);
      pares.forEach((p,i)=>marker(ctx,p.pixel[0]*cv.width/ORIGINAL_W,p.pixel[1]*cv.height/ORIGINAL_H,i+1));
      if(pendiente)marker(ctx,pendiente[0]*cv.width/ORIGINAL_W,pendiente[1]*cv.height/ORIGINAL_H,pares.length+1);
    }
    if(pi.complete && pi.naturalWidth && cp.width){const ctx=cp.getContext('2d');ctx.drawImage(pi,0,0);
      pares.forEach((p,i)=>marker(ctx,PAD+p.field[0]*SCALE,PAD+p.field[1]*SCALE,i+1));
    }
    finish.disabled=pares.length<4 || pendiente!==null;
    estado.textContent='Pares: '+pares.length+' | '+(pendiente ? 'Ahora clic en el mismo punto del PLANO VERDE.' : 'Ahora clic en un nuevo punto del FOTOGRAMA.');
  }
  cv.addEventListener('click',e=>{
    if(pendiente || !cv.width)return;
    let [x,y]=pos(e,cv);
    pendiente=[x*ORIGINAL_W/cv.width,y*ORIGINAL_H/cv.height];render();
  });
  cp.addEventListener('click',e=>{
    if(!pendiente || !cp.width)return;
    let [x,y]=pos(e,cp);
    const xm=(x-PAD)/SCALE,ym=(y-PAD)/SCALE;
    if(xm<0 || xm>LARGO || ym<0 || ym>ANCHO){alert('Elige un punto DENTRO de la cancha.');return;}
    pares.push({pixel:pendiente,field:[xm,ym]});pendiente=null;render();
  });
  undo.addEventListener('click',()=>{pendiente=null;pares.pop();render();});
  reset.addEventListener('click',()=>{pendiente=null;pares=[];render();});
  finish.addEventListener('click',()=>{if(pares.length>=4&&!pendiente){root.remove();resolve(pares);}});
})"""
for key,val in {
    '__URL_FRAME__':url_frame,'__URL_PLANO__':url_plano,
    '__ORIGINAL_W__':str(ancho_original),'__ORIGINAL_H__':str(alto_original),
    '__PAD__':str(MARGEN_PLANO),'__SCALE__':str(ESCALA_PLANO),
    '__LARGO__':str(LARGO_M),'__ANCHO__':str(ANCHO_M)
}.items():
    js=js.replace(key,val)
PARES = eval_js(js,timeout_sec=1800)
print(f'Capturados {len(PARES)} pares de puntos.')
for i,p in enumerate(PARES,1):print(i,'pixel:',[round(v,1) for v in p['pixel']],'campo (m):',[round(v,2) for v in p['field']])

**Alternativa si la interfaz de clics no funciona:** sustituye `PARES` por una lista con este formato, usando las coordenadas reales de tus landmarks. Puedes obtener píxeles mostrando el fotograma con ejes en Matplotlib y leyendo la escala. **Estos valores son solo ejemplos de estructura, NO datos válidos de tu video**:

```python
PARES = [
  {'pixel':[px1,py1], 'field':[X1,Y1]},
  {'pixel':[px2,py2], 'field':[X2,Y2]},
  {'pixel':[px3,py3], 'field':[X3,Y3]},
  {'pixel':[px4,py4], 'field':[X4,Y4]},
]
```

In [ ]:
# Matriz H: transforma (pixel_x, pixel_y) -> (metros_x, metros_y).
if len(PARES)<4:
    raise ValueError('Se necesitan >=4 pares de referencias.')
pixeles=np.asarray([p['pixel'] for p in PARES],dtype=np.float64)
metros=np.asarray([p['field'] for p in PARES],dtype=np.float64)

if cv2.contourArea(cv2.convexHull(pixeles.astype('float32')))<100:
    raise ValueError('Puntos del fotograma demasiado alineados o muy juntos.')
if cv2.contourArea(cv2.convexHull(metros.astype('float32')))<1:
    raise ValueError('Puntos de cancha demasiado alineados o juntos.')

metodo=cv2.RANSAC if len(PARES)>=5 else 0
H, mascara=cv2.findHomography(pixeles,metros,metodo, ransacReprojThreshold=1.5)
if H is None or not np.isfinite(H).all() or abs(np.linalg.det(H))<1e-12:
    raise ValueError('No se pudo determinar una homografía estable. Elige más puntos, no colineales.')

proyeccion=cv2.perspectiveTransform(pixeles.astype('float32').reshape(-1,1,2),H).reshape(-1,2)
errores=np.linalg.norm(proyeccion-metros,axis=1)
print('H (píxeles -> metros):')
print(H)
print('Error de ajuste por referencia (m):',np.round(errores,2))
print('RMSE de ajuste (m):',round(float(np.sqrt(np.mean(errores**2))),3))
if len(PARES)==4:
    print('AVISO: con solo 4 puntos, el error de ajuste puede ser casi cero incluso con referencias equivocadas.')
    print('Valida usando AL MENOS un quinto landmark conocido que no se usó para calibrar.')
else:
    print('Inliers RANSAC:',int(mascara.sum()),'/',len(PARES))

# Prueba más honesta cuando hay >=5 referencias: reservar la última como punto independiente.
if len(PARES)>=5:
    H_test,_=cv2.findHomography(pixeles[:-1],metros[:-1], cv2.RANSAC if len(PARES)>5 else 0, ransacReprojThreshold=1.5)
    if H_test is not None:
        est=cv2.perspectiveTransform(pixeles[-1:].astype('float32').reshape(-1,1,2),H_test).reshape(-1,2)[0]
        error_test=float(np.linalg.norm(est-metros[-1]))
        print(f'Error independiente del último landmark: {error_test:.2f} m')
        if error_test>2:
            print('ALERTA: revisar correspondencias. Error independiente >2 m (umbral práctico, no norma universal).')

In [ ]:
# Proyectar los jugadores del mismo fotograma de calibración.
tracks_frame=tracks[tracks.frame==frame_idx].copy()
if tracks_frame.empty:
    raise ValueError('Este frame no aparece en el CSV. Verifica FPS, video y CSV.')

# Punto de contacto aproximado con el suelo = centro inferior de la bounding box
pies_px=np.column_stack(((tracks_frame.x1.to_numpy()+tracks_frame.x2.to_numpy())/2,
                          tracks_frame.y2.to_numpy())).astype('float32')
pies_m=cv2.perspectiveTransform(pies_px.reshape(-1,1,2),H).reshape(-1,2)
tracks_frame['x_m']=pies_m[:,0]
tracks_frame['y_m']=pies_m[:,1]
tracks_frame['en_cancha']=(tracks_frame.x_m.between(0,LARGO_M) & tracks_frame.y_m.between(0,ANCHO_M))

# Plano con jugadores, dentro del campo.
img=plano_bgr.copy()
for p in tracks_frame.itertuples():
    if not p.en_cancha:continue
    x,y=int(round(MARGEN_PLANO+p.x_m*ESCALA_PLANO)),int(round(MARGEN_PLANO+p.y_m*ESCALA_PLANO))
    cv2.circle(img,(x,y),5,(35,45,255),-1)
    cv2.putText(img,str(p.track_id),(x+5,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1,cv2.LINE_AA)

fig,ax=plt.subplots(1,2,figsize=(18,7))
frame_anotado=frame_bgr.copy()
for p in tracks_frame.itertuples():
    foot=(int(round((p.x1+p.x2)/2)),int(p.y2))
    cv2.circle(frame_anotado,foot,4,(0,255,255),-1)
ax[0].imshow(cv2.cvtColor(frame_anotado,cv2.COLOR_BGR2RGB))
ax[0].set_title('Pies aproximados: mismo frame de calibración');ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
ax[1].set_title('Proyección en metros | puntos fuera del césped omitidos');ax[1].axis('off')
plt.tight_layout();plt.show()

print('Tracks visibles en el frame:',len(tracks_frame))
print('Puntos proyectados dentro de la cancha:',int(tracks_frame.en_cancha.sum()))
print(tracks_frame[['track_id','x_m','y_m','en_cancha']].round(2).to_string(index=False))

In [ ]:
import json
nombre=VIDEO.stem
path_json=SALIDA/f'{nombre}_homografia.json'
path_csv=SALIDA/f'{nombre}_posiciones_frame_{frame_idx}.csv'
path_png=SALIDA/f'{nombre}_minimapa_frame_{frame_idx}.png'

info={
  'video':str(VIDEO),
  'csv_de_tracks':str(CSV_TRACKS),
  'fps':fps,
  'frame_calibrado':int(frame_idx),
  'tiempo_calibrado_s':frame_idx/fps,
  'largo_cancha_m':LARGO_M,
  'ancho_cancha_m':ANCHO_M,
  'H_pixel_a_metros':H.tolist(),
  'referencias':PARES,
  'rmse_ajuste_m':float(np.sqrt(np.mean(errores**2))),
  'advertencia':'H VALIDA SOLO para el frame calibrado o un intervalo SIN movimiento apreciable de cámara.'
}
path_json.write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf-8')
tracks_frame.to_csv(path_csv,index=False)
cv2.imwrite(str(path_png),img)
print('Archivos creados:')
print(path_json)
print(path_csv)
print(path_png)
print('\nNO utilices una H fija para todo el clip si hay paneo, zoom o corte de cámara.')

## Siguiente fase de FUT.AI

Cuando confirmemos que los puntos del minimapa coinciden **con posiciones reales plausibles**:

1. Validar la homografía con uno o más landmarks no usados para su ajuste.
2. Para clips con cámara móvil, **estimar o actualizar H por fotograma o por segmento estable** con referencias de cancha (keypoints, líneas). Si no hay referencias suficientes, marcar el fotograma como *sin calibración fiable*.
3. Solo entonces generar series `(track_id, tiempo_s, x_m, y_m)` y calcular distancia/velocidad con manejo de IDs interrumpidos, saltos anómalos y tiempos reales.

**No confundas el control de cámara de BoT-SORT con calibración del campo:** GMC puede mejorar la asociación de identidades, pero no produce por sí solo posiciones en metros.